# 🎙️🎥 AVSR — обучение на MUAVIC

**Датасет**: MUAVIC (Multilingual Audio-Visual Corpus, Facebook Research)  
**Английская часть**: ~8–10 часов TED-докладов, открытый доступ  
**Репо**: https://github.com/facebookresearch/muavic

## Что делает этот ноутбук
1. Проверяет GPU
2. Монтирует Google Drive (для персистентного хранения данных и чекпоинтов)
3. Устанавливает зависимости
4. Скачивает MUAVIC английскую часть (~3–5 ГБ)
5. Запускает `prepare_data.py` → `.npy` кропы губ + JSONL манифесты
6. Тренирует AVSR-модель (Whisper encoder + видео-ветка + cross-attention)
7. Оценивает WER/CER на тест-сете

## Время
- Первый запуск (скачивание + препроцессинг): ~2–4 часа
- Обучение 10 эпох: ~6–8 часов на T4
- Повторный запуск (данные уже на Drive): ~6–8 часов (только обучение)

> **Важно**: включи T4 GPU в Runtime → Change runtime type → GPU

In [ ]:
# ── 1. Проверка GPU ──────────────────────────────────────────────────────────
import torch
print('PyTorch  :', torch.__version__)
print('CUDA     :', torch.version.cuda)
print('GPU      :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '⚠️  НЕТ GPU!')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM     : {vram:.1f} GB')
    assert vram > 10, 'VRAM < 10 GB — включи T4 GPU в Runtime → Change runtime type!'
    print('✅ GPU готов')
else:
    raise RuntimeError('GPU не найден — включи T4 GPU!')

In [ ]:
# ── 2. Google Drive ──────────────────────────────────────────────────────────
# Монтируем Drive для персистентного хранения.
# ДАННЫЕ и ЧЕКПОИНТЫ будут здесь — пережившают перезапуск сессии Colab.
from google.colab import drive
import os

drive.mount('/content/drive')

BASE      = '/content/drive/MyDrive/avsr_cursach'
CODE_DIR  = f'{BASE}/code'          # исходники проекта
DATA_DIR  = f'{BASE}/muavic_data'   # сырые данные MUAVIC
PROC_DIR  = f'{BASE}/processed'     # .npy кропы губ
MANIF_DIR = f'{BASE}/manifests'     # train.jsonl, val.jsonl, test.jsonl
CKPT_DIR  = f'{BASE}/checkpoints'   # чекпоинты модели

for d in [CODE_DIR, DATA_DIR, PROC_DIR, MANIF_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✅ Drive подключён')
print(f'   Данные:      {DATA_DIR}')
print(f'   Чекпоинты:   {CKPT_DIR}')

In [ ]:
# ── 3. Зависимости ──────────────────────────────────────────────────────────
# PyTorch/torchaudio/torchvision уже есть в Colab.
# Доставляем только то, чего нет.
!pip install -q \
    transformers==4.41.2 \
    omegaconf==2.3.0 \
    jiwer==3.0.4 \
    soundfile==0.12.1 \
    'mediapipe>=0.10.18' \
    opencv-python-headless \
    einops==0.8.0 \
    av

print('✅ Зависимости установлены')

In [ ]:
# ── 4. Загрузка кода проекта ─────────────────────────────────────────────────
import os, sys

# Вариант A: git clone из GitHub (замени URL на свой репозиторий)
# REPO_URL = 'https://github.com/YOUR_ACCOUNT/cursera_claude.git'
# if not os.path.exists(os.path.join(CODE_DIR, 'src')):
#     !git clone $REPO_URL $CODE_DIR

# Вариант B: загрузить avsr_src.zip на Drive и распаковать
import zipfile
zip_path = f'{BASE}/avsr_src.zip'
if os.path.exists(zip_path) and not os.path.exists(os.path.join(CODE_DIR, 'src')):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(CODE_DIR)
    print(f'✅ Код распакован из {zip_path}')

# Добавляем корень проекта в PYTHONPATH
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# Проверка
from src.models.avsr_model import build_model
print(f'✅ Код загружен из {CODE_DIR}')

In [ ]:
# ── 5. Скачивание MUAVIC ─────────────────────────────────────────────────────
#
# MUAVIC — Multilingual Audio-Visual Corpus (Meta / Facebook Research)
# Бумага: Anwar et al., "MuAViC: A Multilingual Audio-Visual Corpus
#         for Robust Speech Recognition and Body Gesture Recognition", 2023
# GitHub: https://github.com/facebookresearch/muavic
#
# Английская часть (~8–10 ч) — TED-доклады из мTEDx.
# Лицензия: Creative Commons Attribution-NonCommercial 4.0.
#
# ВНИМАНИЕ: скачивание занимает ~30-60 минут.
# Данные скачиваются на Drive — повторный запуск ноутбука пропустит этот шаг.

import os

# Клонируем репозиторий MUAVIC (только скрипты, ~1 МБ)
MUAVIC_REPO = '/content/muavic'
if not os.path.exists(MUAVIC_REPO):
    !git clone --depth 1 https://github.com/facebookresearch/muavic.git $MUAVIC_REPO

# Устанавливаем зависимости MUAVIC (yt-dlp, ffmpeg и т.д.)
!pip install -q yt-dlp
!apt-get -qq install -y ffmpeg

# Проверяем, не скачано ли уже
en_train = os.path.join(DATA_DIR, 'en', 'train')
if os.path.exists(en_train) and len(os.listdir(en_train)) > 10:
    n = len(os.listdir(en_train))
    print(f'✅ MUAVIC/en/train уже скачан: {n} папок — пропускаем')
else:
    print('🔽 Скачиваю MUAVIC английскую часть...')
    print('   Это займёт 30-60 минут. Не закрывай браузер!')
    # Используем скрипт из MUAVIC репозитория
    !cd $MUAVIC_REPO && python download.py \
        --data-dir {DATA_DIR} \
        --subset en \
        2>&1 | tail -20
    print('✅ MUAVIC скачан')

# Показываем структуру
print('\nСтруктура данных:')
!find {DATA_DIR}/en -maxdepth 2 -type d | head -20

In [ ]:
# ── 5b. Проверка структуры MUAVIC ────────────────────────────────────────────
# MUAVIC хранит данные в формате:
#   en/
#     train/
#       SPEAKER_ID/
#         UTERANCE_ID.mp4   ← видео со звуком
#     valid/
#     test/
#   en_text/
#     train.tsv  ← ID \t транскрипция
#     valid.tsv
#     test.tsv

import os
from pathlib import Path

def count_samples(split):
    p = Path(DATA_DIR) / 'en' / split
    if not p.exists():
        return 0
    return sum(1 for f in p.rglob('*.mp4'))

for split in ['train', 'valid', 'test']:
    n = count_samples(split)
    print(f'{split:8s}: {n:5d} видео')

# Проверяем формат транскрипций
tsv = Path(DATA_DIR) / 'en_text' / 'train.tsv'
if tsv.exists():
    with open(tsv) as f:
        lines = f.readlines()[:3]
    print(f'\nПервые 3 строки train.tsv:')
    for l in lines:
        print(' ', l.strip())
else:
    # Альтернативный формат: txt файлы рядом с mp4
    print('tsv не найден, ищем txt файлы...')
    txts = list((Path(DATA_DIR) / 'en' / 'train').rglob('*.txt'))
    print(f'txt файлов в train: {len(txts)}')
    if txts:
        print('Пример:', open(txts[0]).read().strip()[:80])

In [ ]:
# ── 6. Сборка JSONL-манифестов из MUAVIC ────────────────────────────────────
# Строим манифест ПЕРЕД препроцессингом: сначала список файлов,
# потом препроцессинг по этому списку.

import json
import soundfile as sf
import subprocess
from pathlib import Path

def get_audio_duration(video_path: str) -> float:
    """Получает длительность через ffprobe (быстро, без декодирования)."""
    try:
        r = subprocess.run(
            ['ffprobe', '-v', 'quiet', '-show_entries', 'format=duration',
             '-of', 'default=noprint_wrappers=1:nokey=1', video_path],
            capture_output=True, text=True
        )
        return float(r.stdout.strip())
    except:
        return -1.0

def build_manifest_from_muavic(split: str, out_path: str,
                                 min_dur=0.5, max_dur=15.0, limit=None):
    """Строит JSONL-манифест из MUAVIC split (train/valid/test)."""
    data_root = Path(DATA_DIR) / 'en'
    text_root = Path(DATA_DIR) / 'en_text'

    # Загружаем транскрипции
    transcripts = {}

    # Формат 1: TSV файл
    tsv_path = text_root / f'{split}.tsv'
    if tsv_path.exists():
        with open(tsv_path) as f:
            for line in f:
                parts = line.strip().split('\t', 1)
                if len(parts) == 2:
                    transcripts[parts[0]] = parts[1].lower().strip()
        print(f'  Загружено {len(transcripts)} транскрипций из TSV')

    # Формат 2: txt файлы рядом с видео
    split_dir = data_root / split
    if not transcripts:
        for txt_path in split_dir.rglob('*.txt'):
            utt_id = txt_path.stem
            transcripts[utt_id] = open(txt_path).read().lower().strip()
        print(f'  Загружено {len(transcripts)} транскрипций из TXT')

    # Находим все mp4 файлы
    video_files = sorted(split_dir.rglob('*.mp4'))
    if limit:
        video_files = video_files[:limit]
    print(f'  Найдено {len(video_files)} mp4 файлов')

    records = []
    skipped = {'no_text': 0, 'duration': 0}

    for vp in video_files:
        utt_id = vp.stem
        # Ищем транскрипцию
        text = transcripts.get(utt_id) or transcripts.get(str(vp.relative_to(split_dir).with_suffix('')))
        if not text:
            skipped['no_text'] += 1
            continue

        dur = get_audio_duration(str(vp))
        if dur < min_dur or dur > max_dur:
            skipped['duration'] += 1
            continue

        records.append({
            'id': utt_id,
            'video': str(vp),
            'audio': str(vp),  # аудио берём из видео-файла
            'lip_npy': '',     # заполнится после preprocessing
            'text': text,
            'duration': round(dur, 3)
        })

    with open(out_path, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')

    print(f'  {out_path}: {len(records)} примеров')
    print(f'  Пропущено: {skipped}')
    return len(records)

# Строим манифесты
for split, fname in [('train', 'train_raw.jsonl'), ('valid', 'val_raw.jsonl'), ('test', 'test_raw.jsonl')]:
    out = os.path.join(MANIF_DIR, fname)
    if os.path.exists(out):
        with open(out) as f:
            n = sum(1 for _ in f)
        print(f'{split}: манифест уже есть ({n} примеров)')
    else:
        print(f'Строю {split}...')
        build_manifest_from_muavic(split, out)

print('\n✅ Манифесты (raw) готовы')

In [ ]:
# ── 7. Препроцессинг: извлечение кропов губ (MediaPipe) ──────────────────────
#
# Это САМЫЙ долгий шаг: ~15-20 секунд на видео на CPU.
# При ~1000 видео — ~4-6 часов.
#
# Прогресс сохраняется в PROC_DIR: если сессия оборвётся —
# просто перезапусти ячейку, уже готовые файлы будут пропущены.
#
# После препроцессинга обновляем манифесты: добавляем lip_npy пути.

import subprocess, json, os
from pathlib import Path

def run_preprocessing(split_raw_manifest: str, split_processed_manifest: str,
                       lips_dir: str, num_workers: int = 2):
    """Запускает prepare_data.py для обработки манифеста."""
    script = os.path.join(CODE_DIR, 'scripts', 'prepare_data.py')
    cmd = [
        'python', script,
        '--manifest-in', split_raw_manifest,
        '--manifest-out', split_processed_manifest,
        '--dst', lips_dir,
        '--num-workers', str(num_workers),
    ]
    print(f'Запускаю: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=False)
    return result.returncode

LIPS_DIR = os.path.join(PROC_DIR, 'lips')
os.makedirs(LIPS_DIR, exist_ok=True)

for split, raw, out in [
    ('train', os.path.join(MANIF_DIR, 'train_raw.jsonl'), os.path.join(MANIF_DIR, 'train.jsonl')),
    ('valid', os.path.join(MANIF_DIR, 'val_raw.jsonl'),   os.path.join(MANIF_DIR, 'val.jsonl')),
    ('test',  os.path.join(MANIF_DIR, 'test_raw.jsonl'),  os.path.join(MANIF_DIR, 'test.jsonl')),
]:
    if not os.path.exists(raw):
        print(f'⚠️  {raw} не найден, пропускаем')
        continue

    if os.path.exists(out):
        with open(out) as f:
            n = sum(1 for _ in f)
        print(f'{split}: манифест уже обработан ({n} примеров)')
        continue

    print(f'\n=== Препроцессинг {split} ===')
    code = run_preprocessing(raw, out, LIPS_DIR, num_workers=2)
    if code == 0:
        print(f'✅ {split} готов')
    else:
        print(f'❌ Ошибка в {split} (код {code})')

print('\n✅ Препроцессинг завершён')

In [ ]:
# ── 8. Конфигурация обучения ─────────────────────────────────────────────────
from omegaconf import OmegaConf

cfg = OmegaConf.create({
    'experiment': {
        'name': 'avsr_muavic_en',
        'seed': 42,
        'output_dir': CKPT_DIR
    },
    'data': {
        'train_manifest': os.path.join(MANIF_DIR, 'train.jsonl'),
        'val_manifest':   os.path.join(MANIF_DIR, 'val.jsonl'),
        'max_duration': 15.0,
        'min_duration': 0.5,
        'num_workers': 2,
        'batch_size': 4,
        'grad_accum': 4,     # эффективный батч = 16
    },
    'audio': {
        'sample_rate': 16000,
        'n_mels': 80,
        'hop_length': 160,
        'win_length': 400
    },
    'video': {'fps': 25, 'lip_size': 96},
    'model': {
        'mode': 'av',                     # ← главный режим: аудио + видео
        'd_model': 512,
        'modality_dropout': 0.1,
        'audio_encoder': {
            'name': 'openai/whisper-small',
            'freeze': True,
            'd_audio': 768
        },
        'video_encoder': {
            'd_video': 512,
            'n_layers': 4,
            'n_heads': 8
        },
        'fusion': {
            'type': 'cross_attention',    # ← основной вариант
            'n_layers': 2,
            'n_heads': 8,
            'dropout': 0.1
        },
        'vocab_size': 29
    },
    'training': {
        'optimizer': 'adamw',
        'lr': 1e-4,
        'weight_decay': 0.01,
        'warmup_steps': 500,
        'max_epochs': 20,
        'patience': 5,
        'grad_clip': 5.0,
        'mixed_precision': True,
        'grad_accum': 4,
        'label_smoothing': 0.0
    },
    'augmentation': {
        'audio': {
            'spec_augment': True,
            'freq_mask': 27,
            'time_mask': 10
        }
    },
    'logging': {
        'use_tensorboard': True,
        'use_wandb': False,
        'log_every_n_steps': 20,
        'val_every_n_epochs': 1
    }
})

print('Конфиг:')
print(OmegaConf.to_yaml(cfg))

In [ ]:
# ── 9. Сборка модели и проверка ──────────────────────────────────────────────
import torch, random
from src.data.dataset import CharTokenizer, AVSRDataset
from src.data.collate import avsr_collate_fn
from src.models.avsr_model import build_model
from torch.utils.data import DataLoader

torch.manual_seed(42); random.seed(42)
device = torch.device('cuda')

tokenizer = CharTokenizer()
model = build_model(cfg, vocab_size=tokenizer.vocab_size).to(device)

n_all = sum(p.numel() for p in model.parameters())
n_tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_frz = n_all - n_tr
print(f'Параметров всего  : {n_all:>12,}')
print(f'  обучаемых       : {n_tr:>12,}  ← видео-ветка + fusion + CTC')
print(f'  заморожено      : {n_frz:>12,}  ← Whisper encoder')

# DataLoaders
train_ds = AVSRDataset(cfg.data.train_manifest, tokenizer, load_video=True, require_lip_cache=True)
val_ds   = AVSRDataset(cfg.data.val_manifest,   tokenizer, load_video=True, require_lip_cache=True)

train_loader = DataLoader(train_ds, batch_size=cfg.data.batch_size,
    shuffle=True, num_workers=cfg.data.num_workers,
    collate_fn=avsr_collate_fn, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=cfg.data.batch_size,
    shuffle=False, num_workers=cfg.data.num_workers,
    collate_fn=avsr_collate_fn, pin_memory=True)

print(f'\nTrain: {len(train_ds):,} примеров | Val: {len(val_ds):,} примеров')

# Sanity check: один батч через модель
batch = next(iter(train_loader))
model.eval()
with torch.no_grad():
    b = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    logits, lens = model(b['audio_mel'], b['audio_lens'], b['video'], b['video_lens'])
print(f'\n✅ Forward pass OK: logits {logits.shape} | out_lens {lens.tolist()}')

In [ ]:
# ── 10. 🚀 ОБУЧЕНИЕ ──────────────────────────────────────────────────────────
#
# Тренер автоматически:
#   - сохраняет best.pt (лучший WER на val) и last.pt каждую эпоху
#   - восстанавливается с last.pt при --resume (на случай обрыва сессии)
#   - логирует loss/WER/LR в TensorBoard
#
# После обрыва: просто перезапусти ячейку — она подхватит last.pt

import logging
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s', datefmt='%H:%M:%S')

from src.training.trainer import Trainer

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,
    cfg=cfg,
    device=device,
    output_dir=CKPT_DIR
)

# Автоматически подхватываем прерванное обучение
last_ckpt = os.path.join(CKPT_DIR, 'last.pt')
if os.path.exists(last_ckpt):
    trainer.load_checkpoint(last_ckpt)
    print(f'🔄 Продолжаю с epoch={trainer.start_epoch}')
else:
    print('🆕 Обучение с нуля')

# ПОЕХАЛИ!
trainer.fit()

In [ ]:
# ── 11. Обучение audio-only baseline ─────────────────────────────────────────
# Для аблации: обучаем ту же модель в режиме audio_only.
# Это наш базовый эксперимент — точка отсчёта WER без видео.

from omegaconf import OmegaConf
from src.models.avsr_model import build_model

cfg_audio = OmegaConf.merge(cfg, OmegaConf.create({
    'model': {'mode': 'audio_only', 'fusion': {'type': 'concat'}},
    'experiment': {'name': 'avsr_audio_only', 'output_dir': CKPT_DIR + '_audio_only'}
}))

model_ao = build_model(cfg_audio, vocab_size=tokenizer.vocab_size).to(device)
trainer_ao = Trainer(model_ao, train_loader, val_loader, tokenizer, cfg_audio, device,
                     output_dir=CKPT_DIR + '_audio_only')

last_ao = CKPT_DIR + '_audio_only/last.pt'
if os.path.exists(last_ao):
    trainer_ao.load_checkpoint(last_ao)

print('Запускаю audio-only baseline...')
trainer_ao.fit()

In [ ]:
# ── 12. Оценка: WER/CER + устойчивость к шуму ───────────────────────────────
import subprocess

best_av    = os.path.join(CKPT_DIR, 'best.pt')
best_ao    = os.path.join(CKPT_DIR + '_audio_only', 'best.pt')
test_manif = os.path.join(MANIF_DIR, 'test.jsonl')
eval_script = os.path.join(CODE_DIR, 'scripts', 'eval.py')

def run_eval(ckpt, mode, snr_list='clean,20,15,10,5,0'):
    cmd = [
        'python', eval_script,
        '--checkpoint', ckpt,
        '--manifest', test_manif,
        '--mode', mode,
        '--noise-snr', snr_list,
        '--output-json', ckpt.replace('.pt', f'_eval_{mode}.json')
    ]
    print(' '.join(cmd))
    subprocess.run(cmd)

# Оцениваем обе модели
if os.path.exists(best_av):
    print('\n=== Оценка AV модели ===')
    run_eval(best_av, 'av')

if os.path.exists(best_ao):
    print('\n=== Оценка Audio-only baseline ===')
    run_eval(best_ao, 'audio_only')

In [ ]:
# ── 13. Вывод результатов ────────────────────────────────────────────────────
import json

def print_results(json_path, label):
    if not os.path.exists(json_path):
        print(f'{label}: файл не найден')
        return
    with open(json_path) as f:
        r = json.load(f)
    print(f'\n{label}:')
    if isinstance(r, dict):
        for k, v in r.items():
            if isinstance(v, (int, float)):
                print(f'  {k:20s}: {v:.4f}')
            else:
                print(f'  {k:20s}: {v}')

print_results(best_av.replace('.pt', '_eval_av.json'), 'AV (cross-attention)')
print_results(best_ao.replace('.pt', '_eval_audio_only.json'), 'Audio-only baseline')

print('\n=== ТАБЛИЦА ДЛЯ КУРСОВОЙ (заполни числами) ===')
print('| Модель           | WER чистое | WER SNR=5 | WER SNR=0 |')
print('|------------------|------------|-----------|-----------|')
print('| Audio-only       |            |           |           |')
print('| AV (concat)      |            |           |           |')
print('| AV (cross-att)   |            |           |           |')

In [ ]:
# ── 14. TensorBoard ──────────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir {CKPT_DIR}

In [ ]:
# ── 15. Статус в любой момент ────────────────────────────────────────────────
import torch, os
from pathlib import Path

print('=== СТАТУС ОБУЧЕНИЯ ===')
for label, ckpt_dir in [
    ('AV (cross-attention)', CKPT_DIR),
    ('Audio-only baseline', CKPT_DIR + '_audio_only'),
]:
    for name in ['best.pt', 'last.pt']:
        p = Path(ckpt_dir) / name
        if p.exists():
            ckpt = torch.load(str(p), map_location='cpu', weights_only=True)
            print(f'[{label}] [{name}] '
                  f'epoch={ckpt.get("epoch","?"):>3}  '
                  f'step={ckpt.get("global_step","?"):>6}  '
                  f'best_wer={ckpt.get("best_wer", float("nan")):.4f}')
        else:
            print(f'[{label}] [{name}] — не найден')

## Инструкция: что делать если что-то пошло не так

### Сессия оборвалась
1. Открой ноутбук снова
2. Запусти ячейки 1–4 (GPU + Drive + deps + код)
3. Запусти ячейку 8 (конфиг)
4. Запусти ячейку 9 (модель) — она сама подхватит `last.pt`
5. Запусти ячейку 10 — обучение продолжится с остановленного места

### MUAVIC скачался с ошибками
1. Проверь, что на Drive достаточно места (нужно ~5 ГБ)
2. Перезапусти ячейку 5 — она умеет докачивать
3. Если совсем не качается, попробуй через VPN

### Ошибка CUDA out of memory
- Уменьши `batch_size` до 2 (и увеличь `grad_accum` до 8 для сохранения эффективного батча)
- Или отключи видео (`mode: audio_only`) для проверки

### MediaPipe не нашёл лицо на многих кадрах
- Это норма для TED-докладов где говорящий иногда отворачивается
- До 20% пропущенных кадров не критично — кропы паддятся нулями
- Если пропусков > 50% — скорее всего проблема с самим видеофайлом